# iNaturalist other-plant observations — Coimbra district

**Purpose.** Download, look at and clean the plant observations that
are not *Acacia*. They become the other-plant classes: they were made
by the same people, in the same places, as the *Acacia* presences, so
the model learns *Acacia* vs. other vegetation rather than "visited
vs. not visited" (Phillips et al., 2009).

**Data.** Every plant observation in the iNaturalist place "Coimbra,
PT" (the district), excluding *Acacia*, all years
(`data/raw/citizen_science/`, see `data/README.md`).

**Download:** 1. observations; 2. obscured coordinates.

**Part 1 — exploration:** 3. when; 4. presences vs. other plants
(map); 5. footprint overlap on a 1 km grid.

**Part 2 — cleaning:** 6. keep dated, non-obscured observations;
7. anonymise the main observer; 8. keep only research grade;
9. positional accuracy; 10. remove other plants in pixels with
*A. dealbata*; 11. burned-area flag; 12. save.

Needs `acacia_observations.ipynb` to have run first (it reads the
cleaned *Acacia* file to compare).

## 0. Setup

- `PLACE_ID = 8422`: the iNaturalist place for the Coimbra district;
- `ACACIA = 47452` and `PLANTAE = 47126`: iNaturalist taxon ids for
  the genus *Acacia* and for all plants;
- `REF_YEAR = 2023`: the year all observations are treated as;
- EPSG:3763 (PT-TM06, metres), so distances and 1 km cells are
  correct.

In [ ]:
import sys
import time
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from shapely.geometry import shape

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from src.burned_areas import burned_between, load_burned_areas
from src.grid import pixel_id

RAW_CS = ROOT / "data" / "raw" / "citizen_science"
INTERIM_CS = ROOT / "data" / "interim" / "citizen_science"
PLACE_ID = 8422          # iNaturalist "Coimbra, PT" (district)
ACACIA = 47452           # genus Acacia
PLANTAE = 47126          # kingdom Plantae
REF_YEAR = 2023          # all observations are treated as this year

place = requests.get(f"https://api.inaturalist.org/v1/places/{PLACE_ID}", timeout=60).json()["results"][0]
district = gpd.GeoSeries([shape(place["geometry_geojson"])], crs=4326).to_crs(3763).iloc[0]

def base(ax, title):
    gpd.GeoSeries([district], crs=3763).boundary.plot(ax=ax, color="black", linewidth=0.8)
    ax.set_title(title); ax.set_axis_off()

## 1. Download: observations

**What it does.** Asks the iNaturalist API for every plant observation
in the Coimbra district (`place_id=8422`), excluding *Acacia*, and
saves them as CSV files in `data/raw/citizen_science/`.

**How.**
- The download is split into three date ranges (before 2021,
  2021–2025, from 2026), because 2021–2025 was fetched first and the
  other years later. Each range is one file.
- The API returns 200 observations per page. Pages are requested in
  order of id, waiting 1 s between requests (iNaturalist's rate
  limit).
- It is **resumable**: each page is appended to a `.partial.csv` file,
  and if the download stops it restarts from the last saved id. When
  a range finishes, the partial file is renamed.
- If a file already exists, it is not downloaded again.

**Output.** A table with the number of observations per year.

In [ ]:
# Downloaded in date ranges; 2021–2025 was fetched first, the other ranges later.
BG_RANGES = {"before_2021": (None, "2020-12-31"),
             "2021_2025": ("2021-01-01", "2025-12-31"),
             "from_2026": ("2026-01-01", None)}
bg_file = lambda key: RAW_CS / f"inaturalist_plantae_no_acacia_coimbra_{key}.csv"

def date_params(key):
    d1, d2 = BG_RANGES[key]
    return {k: v for k, v in dict(d1=d1, d2=d2).items() if v}

def download_background(key):
    """Resumable: appends each page to a partial file and restarts from the last id saved."""
    partial = bg_file(key).with_suffix(".partial.csv")
    url = "https://api.inaturalist.org/v1/observations"
    last_id = int(pd.read_csv(partial, usecols=["id"])["id"].max()) if partial.exists() else 0
    params = dict(place_id=PLACE_ID, taxon_id=PLANTAE, without_taxon_id=ACACIA, **date_params(key),
                  order_by="id", order="asc", per_page=200, id_above=last_id)
    while True:
        r = requests.get(url, params=params, timeout=60)
        r.raise_for_status()
        res = r.json()["results"]
        if not res:
            break
        rows = []
        for o in res:
            if not o.get("geojson"):
                continue
            lon, lat = o["geojson"]["coordinates"]
            t = o.get("taxon") or {}
            rows.append(dict(id=o["id"], observed_on=o.get("observed_on"),
                             taxon_id=t.get("id"), taxon_name=t.get("name"), taxon_rank=t.get("rank"),
                             latitude=lat, longitude=lon,
                             positional_accuracy=o.get("positional_accuracy"),
                             quality_grade=o.get("quality_grade"),
                             user_login=(o.get("user") or {}).get("login"),
                             license_code=o.get("license_code")))
        pd.DataFrame(rows).to_csv(partial, mode="a", header=not partial.exists(), index=False)
        params["id_above"] = res[-1]["id"]
        time.sleep(1)  # iNaturalist asks for <= 60 requests/min
    partial.rename(bg_file(key))

for key in BG_RANGES:
    if not bg_file(key).exists():
        download_background(key)
bg_raw = pd.concat([pd.read_csv(bg_file(key)) for key in BG_RANGES], ignore_index=True)
bg_raw.groupby(pd.to_datetime(bg_raw["observed_on"], errors="coerce").dt.year).size()

**Result.** 120 479 observations. Like *Acacia*, they grow fast
after 2019: most are from 2021–2026.

## 2. Download: obscured coordinates

**What it does.** Downloads the ids of observations whose location is
hidden, and stores them in `*_obscured_ids.csv` (one file per range).

**Why.** When an observer or a species has geoprivacy set to
"obscured" or "private", iNaturalist moves the point to a random spot
inside a ~0.2° cell (several km). Those coordinates are useless for
10 m pixels. They are excluded in step 6.

**How.** Two queries per range, one for user geoprivacy and one for
taxon geoprivacy, returning ids only.

**Output.** The number of obscured ids.

In [ ]:
obscured_file = lambda key: RAW_CS / f"inaturalist_plantae_no_acacia_coimbra_{key}_obscured_ids.csv"

def download_obscured_ids(key):
    url = "https://api.inaturalist.org/v1/observations"
    ids = set()
    for privacy in [dict(geoprivacy="obscured,private"), dict(taxon_geoprivacy="obscured,private")]:
        params = dict(place_id=PLACE_ID, taxon_id=PLANTAE, without_taxon_id=ACACIA, **date_params(key),
                      order_by="id", order="asc", per_page=200, id_above=0, only_id="true", **privacy)
        while True:
            r = requests.get(url, params=params, timeout=60)
            r.raise_for_status()
            res = r.json()["results"]
            if not res:
                break
            ids.update(o["id"] for o in res)
            params["id_above"] = res[-1]["id"]
            time.sleep(1)
    pd.DataFrame({"id": sorted(ids)}).to_csv(obscured_file(key), index=False)

for key in BG_RANGES:
    if not obscured_file(key).exists():
        download_obscured_ids(key)
obscured_ids = set().union(*(set(pd.read_csv(obscured_file(key))["id"]) for key in BG_RANGES))
len(obscured_ids)

**Result.** 9 903 observations (8%) have obscured coordinates.

# Part 1 — Exploration

## 3. When

Observations per year (from the download above). The comparisons below
leave out obscured coordinates, which are displaced by several km.

In [ ]:
bg_raw["date"] = pd.to_datetime(bg_raw["observed_on"], errors="coerce")
bg_raw["year"] = bg_raw["date"].dt.year
print(f"{len(bg_raw):,} observations, {bg_raw.year.min():.0f}–{bg_raw.year.max():.0f}; "
      f"{bg_raw.year.isna().sum()} without a date; {bg_raw.id.isin(obscured_ids).sum():,} obscured")
bg_raw.year.value_counts().sort_index().loc[2010:].plot.bar(color="tab:gray", figsize=(10, 3), title="Other plants per year (from 2010)")
plt.tight_layout()

## 4. Presences vs. other plants

**What:** *A. dealbata* presences (red, cleaned file) on top of other-plant
observations (grey), in three panels: all observers, Observer A only,
and everyone else. Observer A is the person with the most *Acacia*
observations (anonymised).
**Why:** using other plants only works if they come from the same
places as the presences. The map shows whether that holds, and how
much of it depends on Observer A.

In [ ]:
acacia = pd.read_csv(INTERIM_CS / "inat_dealbata_clean.csv")
acacia = gpd.GeoDataFrame(acacia, geometry=gpd.points_from_xy(acacia.longitude, acacia.latitude), crs=4326).to_crs(3763)
bg = bg_raw[bg_raw["year"].notna() & ~bg_raw["id"].isin(obscured_ids)].copy()
bg = gpd.GeoDataFrame(bg, geometry=gpd.points_from_xy(bg.longitude, bg.latitude), crs=4326).to_crs(3763)

# Observer A = the person with the most Acacia observations (username not shown)
acacia_raw = pd.read_csv(RAW_CS / "inaturalist_acacia_coimbra_pilot.csv")
dominant = acacia_raw.loc[~acacia_raw["coordinates_obscured"], "user_login"].value_counts().index[0]
bg["observer"] = np.where(bg["user_login"] == dominant, "Observer A", "Others")

groups = [("All observers", acacia, bg),
          ("Observer A", acacia[acacia.observer == "Observer A"], bg[bg.observer == "Observer A"]),
          ("Others", acacia[acacia.observer == "Others"], bg[bg.observer == "Others"])]
fig, axes = plt.subplots(1, 3, figsize=(18, 7), sharex=True, sharey=True)
for ax, (label, p, b) in zip(axes, groups):
    b.plot(ax=ax, markersize=0.5, color="tab:gray", alpha=0.3, label=f"other plants (n={len(b)})")
    p.plot(ax=ax, markersize=2, color="tab:red", alpha=0.7, label=f"Acacia (n={len(p)})")
    base(ax, label); ax.legend(loc="lower right", markerscale=5)
plt.tight_layout()

**Result.** The other plants (grey) cover the whole district, and the
*Acacia* presences (red) sit inside areas that also have other
plants, for Observer A and for everyone else.

## 5. Footprint overlap on a 1 km grid

**What it does.** Divides the district into 1 km × 1 km cells and, for
each group (all, Observer A, others), counts:
- the cells with *Acacia* and the cells with other plants;
- `pct_acacia_cells_with_background`: of the cells with *Acacia*, the
  share that also have other plants. High = the other plants cover
  the places where *Acacia* was seen;
- `pct_background_cells_with_acacia`: of the cells with other plants,
  the share that also have *Acacia*. Low = other plants cover much
  more territory than *Acacia*;
- `ratio_bg_per_presence`: other-plant observations per *Acacia*
  presence.

In [ ]:
def cells(df, size=1000):
    return set(zip((df.geometry.x // size).astype(int), (df.geometry.y // size).astype(int)))

rows = []
for label, p, b in groups:
    cp, cb = cells(p), cells(b)
    rows.append(dict(group=label, acacia=len(p), background=len(b),
                     ratio_bg_per_presence=round(len(b) / max(len(p), 1), 1),
                     acacia_cells_1km=len(cp), background_cells_1km=len(cb),
                     pct_acacia_cells_with_background=round(100 * len(cp & cb) / max(len(cp), 1), 1),
                     pct_background_cells_with_acacia=round(100 * len(cp & cb) / max(len(cb), 1), 1)))
pd.DataFrame(rows)

**Result.**
- 32 other-plant observations per *A. dealbata* presence.
- 96.6% of the 1 km cells with *A. dealbata* also have other plants
  (97.5% for Observer A, 93.2% for the others): the other plants
  cover the places where *A. dealbata* was seen.
- Only 16.3% of the cells with other plants have *A. dealbata*: the
  other plants cover much more territory.
- Observer A also records other plants: 36 080 observations (33%).

# Part 2 — Cleaning (preprocessing)

The same steps as for *Acacia* (see `acacia_observations.ipynb`),
saved to `data/interim/citizen_science/inat_other_plants_clean.csv`.

Not done yet (open decisions): group species into classes, then keep
one observation per pixel and class.

## 6. Keep dated, non-obscured observations

Already applied above when building `bg` (section 4): observations
without a date and with obscured coordinates are left out.

In [ ]:
print(f"{len(bg_raw):,} → {len(bg):,} (dropped {bg_raw.year.isna().sum()} without a date, "
      f"{bg_raw.id.isin(obscured_ids).sum():,} obscured)")

## 7. Anonymise the main observer

Done in section 4: `observer` is "Observer A" (the same person as in
the *Acacia* file) or "Others"; the username is not saved.

In [ ]:
bg["observer"].value_counts()

## 8. Keep only research grade

**What:** keep observations with `quality_grade = research`; drop
`casual` and `needs_id`.
**Why (decided 2026-10-29, see `thesis/notes.md`):** the same rule as
for the *A. dealbata* presences (`acacia_observations.ipynb`, step
11). Other plants only work as a comparison if they were recorded by
the same process as the presences (target-group background, Phillips
et al., 2009). With research grade on one side only, the model could
learn how observations were made (photo-less trips, cultivated garden
plants) instead of the plant.

**Future work:** the sensitivity check with and without `casual` and
`needs_id` is run on both sides at once. Many `needs_id` are
identified to genus, which may be enough to group them into classes:
they can be recovered from the raw files if a class lacks data.

In [ ]:
print(pd.crosstab(bg["quality_grade"], bg["observer"], margins=True))
before = len(bg)
bg = bg[bg["quality_grade"] == "research"]
print(f"\n{before:,} → {len(bg):,} (dropped {before - len(bg):,} casual or needs_id)")
n_pres = len(acacia)
print(f"other plants per A. dealbata presence: {len(bg) / n_pres:.1f}; Observer A: {(bg.observer == 'Observer A').mean():.1%}")

# Does the footprint still match after the filter? (same measures as section 5)
for label, p, b in [("All observers", acacia, bg),
                    ("Observer A", acacia[acacia.observer == "Observer A"], bg[bg.observer == "Observer A"]),
                    ("Others", acacia[acacia.observer == "Others"], bg[bg.observer == "Others"])]:
    cp, cb = cells(p), cells(b)
    print(f"{label}: {100 * len(cp & cb) / len(cp):.1f}% of A. dealbata cells have other plants; "
          f"{100 * len(cp & cb) / len(cb):.1f}% of other-plant cells have A. dealbata")

**Result.** 60 630 research-grade observations remain; 49 946 are
dropped (12 458 `casual`, 37 488 `needs_id`). There are still 20.2
other-plant observations per (final) *A. dealbata* presence, and the
footprint still matches: 94.3% of the 1 km cells with *A. dealbata*
have other plants (96.7% for Observer A, 88.3% for the others).
Observer A made 29.0% of them.

## 9. Positional accuracy

**What:** the same rule as for the presences
(`acacia_observations.ipynb`, step 12, where the test behind it is):
drop observations with a known `positional_accuracy` above 30 m; keep
those without a value, marked with `accuracy_missing`.
**Why the same rule:** as for the quality grade, both sides must go
through the same filters.

In [ ]:
print(f"without a value: {bg.positional_accuracy.isna().mean():.1%} "
      f"(Observer A: {bg.loc[bg.observer == 'Observer A', 'positional_accuracy'].isna().mean():.1%}, "
      f"others: {bg.loc[bg.observer == 'Others', 'positional_accuracy'].isna().mean():.1%})")
before = len(bg)
bg["accuracy_missing"] = bg["positional_accuracy"].isna()
bg = bg[bg["accuracy_missing"] | (bg["positional_accuracy"] <= 30)]
print(f"{before:,} → {len(bg):,} (dropped {before - len(bg):,} with accuracy > 30 m); "
      f"without a value (kept): {bg.accuracy_missing.sum():,}")
print(f"other plants per A. dealbata presence: {len(bg) / len(acacia):.1f}; Observer A: {(bg.observer == 'Observer A').mean():.1%}")

**Result.** As for the presences, missing values are mostly
Observer A's (95.9% of theirs, 19.3% of the others'). 12 505
observations dropped (accuracy > 30 m); 48 125 remain, 25 163 of them
without a value (kept). 16.0 other plants per presence; Observer A's
share rises to 36.2%.

## 10. Remove other plants in pixels with *A. dealbata*

**What:** each observation gets a `pixel` id (same 10 m grid as the
presences, `src/grid.py`). Other plants in a pixel that also holds a
cleaned *A. dealbata* presence are dropped; the presence stays.
**Why (decided 2026-10-29, see `thesis/notes.md`):** otherwise the
same pixel says "*A. dealbata*" and "not *A. dealbata*", which
confuses training. In the field this is normal (other plants under or
next to the acacias). The presence stays because it is the plant to
map and it was seen there. This is the smallest version of the
supervisor's segmentation mask (one pixel); a larger zone around each
acacia waits for his answer.

Repeats within the other plants are left for now: a pixel with a
eucalyptus and an oak holds two different classes, which is only
known after grouping species into classes.

In [ ]:
bg["pixel"] = pixel_id(bg)
conflict = bg["pixel"].isin(set(acacia["pixel"]))
print(f"pixels with both: {bg.loc[conflict, 'pixel'].nunique()}; other plants in them: {conflict.sum()}")
before = len(bg)
bg = bg[~conflict]
print(f"{before:,} → {len(bg):,}")
shared = bg["pixel"].duplicated(keep=False)
print(f"other plants sharing a pixel (left for after the classes): {shared.mean():.1%}; "
      f"other plants per A. dealbata presence: {len(bg) / len(acacia):.1f}")

**Result.** 443 pixels held both; their 772 other-plant observations
are dropped (48 125 → 47 353). 40.6% of the remaining other plants
still share a pixel with another other plant, to be handled after the
classes. 15.8 other plants per presence.

## 11. Burned-area flag

Same rule and code as for *Acacia* (`src/burned_areas.py`): `burned`
is `True` if the observation lies in an area that burned between the
observation and 2023. Nothing is removed.

In [ ]:
fires = load_burned_areas(ROOT / "data" / "external" / "auxiliary" / "icnf_burned_areas")
fires = fires[fires.intersects(district)]
bg["burned"], bg["burned_same_year"] = burned_between(bg, fires, REF_YEAR)
print(f"burned between observation and {REF_YEAR}: {bg.burned.sum():,} ({bg.burned.mean():.1%}); "
      f"of which same year without a fire date: {bg.burned_same_year.sum()}")

**Result.** 522 observations (1.1%) are flagged, more than for
*A. dealbata* (0.2%): other plants include older records, and the
2017 fires burned after many of them. No case is ambiguous.

## 12. Save

In [ ]:
OUT = INTERIM_CS / "inat_other_plants_clean.csv"
keep = ["id", "observed_on", "year", "taxon_id", "taxon_name", "taxon_rank", "latitude", "longitude",
        "positional_accuracy", "quality_grade", "accuracy_missing", "observer", "pixel", "burned", "burned_same_year"]
bg[keep].to_csv(OUT, index=False)
print(OUT.relative_to(ROOT), f"{len(bg):,} observations")

## Summary

**Download.** 120 479 plant observations that are not *Acacia*,
1976–2026; 9 903 with obscured coordinates.

**Part 1 — exploration.** 110 576 dated, non-obscured observations,
32 per *A. dealbata* presence (research grade). They cover the places
where *A. dealbata* was seen (96.6% of its 1 km cells) and much more
(only 16.3% of their cells have *A. dealbata*). Observer A made 33% of
them.

**Part 2 — cleaning.** Obscured coordinates dropped; main observer
anonymised; only research grade kept, the same rule as for the
presences (110 576 → 60 630); positional accuracy above 30 m dropped,
missing values kept and marked, also the same rule (60 630 → 48 125);
other plants in the 443 pixels that also hold an *A. dealbata*
presence dropped (48 125 → 47 353, 15.8 per presence); `burned` flag
added (522, 1.1%; nothing removed). Result: 47 353 observations in
`data/interim/citizen_science/inat_other_plants_clean.csv`. Still
open: group species into classes, then one observation per pixel and
class. Future work: sensitivity check with and without `casual` and
`needs_id`, on both sides.